In [2]:
pip install -U langchain-neo4j

  Using cached pypdf-6.16.2-py3-none-any.whl.metadata (7.5 kB)
Using cached pypdf-6.16.2-py3-none-any.whl (385 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 2.4 MB/s  0:00:15m0:00:0100:01
  Attempting uninstall: pypdf90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [scipy]
    Found existing installation: pypdf 6.8.0━━━━━━━━━━━━━━━━━━ 1/7 [scipy]
    Uninstalling pypdf-6.8.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [scipy]
      Successfully uninstalled pypdf-6.8.0━━━━━━━━━━━━━━━━━━━━ 1/7 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [langchain-neo4j] [langchain-neo4j]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

In [11]:
load_dotenv()

True

In [12]:
# temperature=0 ensures deterministic entity and relationship extraction
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [13]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("data/elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

Page 1: 2343 chars
Page 2: 1192 chars


In [14]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [15]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

In [16]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [18]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

14 graph documents extracted
Nodes: ['Elon Musk', 'Justine Wilson', 'Nevada Alexander Musk', 'Griffin Musk', 'Xavier Musk', 'Damian Musk', 'Saxon Musk', 'Kai Musk', 'Vivian Jenna Wilson']
Rels:  [('Elon Musk', 'BORN_ON', 'June 28, 1971'), ('Elon Musk', 'BORN_IN', 'Pretoria, South Africa'), ('Elon Musk', 'HAS_NATIONALITY', 'American'), ('Elon Musk', 'RECOGNISED_AS', "World'S Wealthiest Person")]


In [19]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

[#BC0A]  _: <CONNECTION> error: Failed to write data to connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Unable to retrieve routing information
Transaction failed and will be retried in 0.8627733919021162s (Unable to retrieve routing information)
[#BC0E]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('p-mt-819cf3afaca3-11-0126.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Transaction failed and will be retried in 2.11209143085935s (Failed to write data to connection IPv4Address(('p-mt-819cf3afaca3-11-0126.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Graph stored in Neo4J


In [20]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [21]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['__Entity__', 'Person'], 'count': 25}
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Location'], 'count': 14}
  {'label': ['__Entity__', 'Product'], 'count': 9}
  {'label': ['__Entity__', 'Company'], 'count': 5}
  {'label': ['__Entity__', 'Organization'], 'count': 5}
  {'label': ['__Entity__'], 'count': 4}
  {'label': ['__Entity__', 'Date'], 'count': 3}
  {'label': ['__Entity__', 'Company', 'Organization'], 'count': 2}
  {'label': ['__Entity__', 'Nationality'], 'count': 1}
  {'label': ['__Entity__', 'Title'], 'count': 1}
  {'label': ['__Entity__', 'Concept'], 'count': 1}
Relationships:
  {'type': 'MENTIONS', 'count': 91}
  {'type': 'PARENT_OF', 'count': 12}
  {'type': 'PARENT', 'count': 12}
  {'type': 'HEADQUARTERED_IN', 'count': 8}
  {'type': 'CO_FOUNDER', 'count': 7}
  {'type': 'MANUFACTURES', 'count': 7}
  {'type': 'ACQUIRED', 'count': 4}
  {'type': 'CEO', 'count': 4}
  {'type': 'LOCATED_IN', 'count': 4}
  {'type': 'PARTNER', 'count': 2}
  {'t